# Phase 2 — Preprocessing & EDA
**Goal:** Load all Ireland JSON files, flatten the nested structure, extract fraud labels,
clean the text, and save a single analysis-ready CSV.

**Output:** `data/contracts_ie_clean.csv`

## 0. Imports & Config

In [ ]:
import json
import re
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt

# Paths — adjust DATA_DIR if your folder is in a different location
DATA_DIR   = "../data-ie-json"
OUTPUT_DIR = "../data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "contracts_ie_clean.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Reading from : {os.path.abspath(DATA_DIR)}")
print(f"Writing to   : {os.path.abspath(OUTPUT_FILE)}")

## 1. Load Raw JSON Files

Each year is a separate JSON file containing an array of contract objects.
We load them all and combine into one list.

In [ ]:
json_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.json")))
print(f"Found {len(json_files)} files:")

all_records = []

for path in json_files:
    filename = os.path.basename(path)
    with open(path, encoding="utf-8") as f:
        records = json.load(f)
    all_records.extend(records)
    print(f"  {filename:40s}  {len(records):>6,} records")

print(f"\nTotal records loaded: {len(all_records):,}")

## 2. Flatten Each Record

Each contract object is nested (buyers, lots, indicators, cpvs are all sub-arrays).
We extract only the fields we need into a plain dictionary — one dict per contract.

### Helper functions

In [ ]:
def get_indicator_value(indicators, indicator_type):
    """
    Search an indicators list for a specific type.
    Returns the numeric value if status is CALCULATED, otherwise None.
    """
    for ind in indicators:
        if ind.get("type") == indicator_type:
            if ind.get("status") == "CALCULATED":
                return ind.get("value")
    return None


def get_ot_score(scores, score_type):
    """Extract a named score from the ot.scores list."""
    for s in scores:
        if s.get("type") == score_type and s.get("status") == "CALCULATED":
            return s.get("value")
    return None


def flatten_record(rec):
    """Convert one raw contract object into a flat dictionary."""
    indicators = rec.get("indicators") or []
    buyers     = rec.get("buyers") or []
    lots       = rec.get("lots") or []
    cpvs       = rec.get("cpvs") or []
    ot         = rec.get("ot") or {}
    ot_scores  = ot.get("scores") or []
    buyer      = buyers[0] if buyers else {}
    lot        = lots[0]   if lots   else {}

    # --- Identifiers & dates ---
    row = {
        "contract_id"      : rec.get("persistentId"),
        "group_id"         : rec.get("groupId"),
        "publication_date" : ot.get("date"),
        "bid_deadline"     : rec.get("bidDeadline"),
        "supply_type"      : rec.get("supplyType"),
        "country"          : rec.get("country"),

        # --- Text fields (cleaned later) ---
        "title"            : rec.get("title") or "",
        "description"      : rec.get("description") or "",

        # --- Buyer info ---
        "buyer_name"       : buyer.get("name"),
        "buyer_id"         : buyer.get("id"),
        "buyer_address"    : (buyer.get("address") or {}).get("rawAddress"),
        "buyer_email"      : buyer.get("email"),
        "buyer_contracts_count" : buyer.get("contractsCount"),
        "buyer_total_value"     : buyer.get("totalValueOfContracts"),

        # --- Lot status ---
        "lot_status"       : lot.get("status"),

        # --- CPV (main procurement category code) ---
        "cpv_main"         : next((c["code"] for c in cpvs if c.get("isMain")), None),

        # --- OpenTender composite scores (0-100) ---
        "score_integrity"     : get_ot_score(ot_scores, "INTEGRITY"),
        "score_transparency"  : get_ot_score(ot_scores, "TRANSPARENCY"),
        "score_tender"        : get_ot_score(ot_scores, "TENDER"),

        # --- Raw fraud indicator values (used to build labels below) ---
        # Value = 100 means the flag is present; 0 means absent.
        "ind_single_bid"         : get_indicator_value(indicators, "INTEGRITY_SINGLE_BID"),
        "ind_advertisement_period": get_indicator_value(indicators, "INTEGRITY_ADVERTISEMENT_PERIOD"),
        "ind_procedure_type"     : get_indicator_value(indicators, "INTEGRITY_PROCEDURE_TYPE"),
        "ind_call_for_tender"    : get_indicator_value(indicators, "INTEGRITY_CALL_FOR_TENDER_PUBLICATION"),
    }

    return row

In [ ]:
rows = [flatten_record(rec) for rec in all_records]
df = pd.DataFrame(rows)

print(f"Shape: {df.shape}")
df.head(2)

## 3. Build Fraud Labels

Labels are derived from the pre-computed OpenTender indicator values:
- `1` = flag present (value = 100)
- `0` = flag absent  (value = 0)
- `NaN` = insufficient data (indicator was not calculable for this record)

**Note on partial scores:** OpenTender sometimes returns values like `75.0` or `87.5`
when a contract has multiple lots and only some triggered the flag. We treat `>= 50`
as flagged (majority of lots suspicious) and `== 0` as clean. Anything in between
that isn't exactly 0 but below 50 stays NaN — there are very few of these.

`winner_concentration` is computed directly from the data:
a buyer in the top 5% by total contracts awarded is flagged.

In [ ]:
def indicator_to_label(value):
    """
    Convert a raw OpenTender indicator value to a binary label.
    - value >= 50  → 1 (flagged: majority of lots triggered the flag)
    - value == 0   → 0 (clean: no lots triggered the flag)
    - value is None or 0 < value < 50 → None (insufficient or ambiguous data)
    """
    if value is None:
        return None
    if value >= 50:
        return 1
    if value == 0:
        return 0
    return None  # partial score below majority threshold — treat as unknown


# single_bid: derived from INTEGRITY_SINGLE_BID
df["single_bid"] = df["ind_single_bid"].apply(indicator_to_label)

# short_tender_period: derived from INTEGRITY_ADVERTISEMENT_PERIOD
df["short_tender_period"] = df["ind_advertisement_period"].apply(indicator_to_label)

# winner_concentration: top 5% buyers by number of contracts awarded
threshold = df["buyer_contracts_count"].quantile(0.95)
df["winner_concentration"] = (df["buyer_contracts_count"] >= threshold).astype(int)

# Summary
print("Label distributions:")
print(f"  {'label':<25} {'flagged':>8} {'clean':>8} {'NaN':>8} {'total':>8}  {'flag_rate'}")
for col in ["single_bid", "short_tender_period", "winner_concentration"]:
    flagged = (df[col] == 1).sum()
    clean   = (df[col] == 0).sum()
    nan     = df[col].isna().sum()
    total   = flagged + clean
    rate    = f"{flagged / total * 100:.1f}%" if total > 0 else "n/a"
    print(f"  {col:<25} {flagged:>8,} {clean:>8,} {nan:>8,} {total:>8,}  {rate}")

## 4. Clean Text Fields

The `title` and `description` fields have three problems we fix here:
1. **Encoding artifacts** — `â€˜` style characters from UTF-8/Latin-1 mismatch
2. **Title repeated at the start of description** — boilerplate duplication
3. **URLs and reference codes** — noise with no semantic value for NLP

In [ ]:
def fix_encoding(text):
    """Repair common UTF-8-read-as-Latin-1 artifacts."""
    if not isinstance(text, str):
        return ""
    try:
        # Re-encode as Latin-1 bytes, then decode correctly as UTF-8
        return text.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text


def clean_text(text):
    """Strip URLs, reference codes, and excess whitespace."""
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r"https?://\S+", "", text)
    # Remove leading reference codes like "JUL120956 - " or "2023/S 045-123456 - "
    text = re.sub(r"^[A-Z0-9/\s\-]{3,30}\s[-–]\s", "", text)
    # Collapse multiple spaces / newlines
    text = re.sub(r"\s+", " ", text).strip()
    return text


def remove_title_prefix(title, description):
    """Remove the title if it appears copy-pasted at the start of the description."""
    if description.startswith(title):
        return description[len(title):].strip()
    return description


# Apply in order: fix encoding → clean → remove title prefix from description
df["title"]       = df["title"].apply(fix_encoding).apply(clean_text)
df["description"] = df["description"].apply(fix_encoding).apply(clean_text)
df["description"] = df.apply(
    lambda r: remove_title_prefix(r["title"], r["description"]), axis=1
)

# Quick sanity check
print("Example title:")
print(" ", df["title"].iloc[0])
print("\nExample description (first 300 chars):")
print(" ", df["description"].iloc[0][:300])

## 5. Drop Unusable Rows

Remove rows that would contribute nothing to the NLP pipeline:
- No text in either `title` or `description`
- Duplicate `contract_id` (keep first occurrence)

In [ ]:
before = len(df)

# Drop rows with no usable text
df = df[df["title"].str.strip().ne("") | df["description"].str.strip().ne("")]

# Drop duplicate contract IDs
df = df.drop_duplicates(subset="contract_id", keep="first")

after = len(df)
print(f"Dropped {before - after:,} rows  ({before:,} → {after:,})")

## 6. Final Column Selection & Type Casting

In [ ]:
# Convert date strings to actual dates
df["publication_date"] = pd.to_datetime(df["publication_date"], errors="coerce")
df["bid_deadline"]     = pd.to_datetime(df["bid_deadline"],     errors="coerce")

# Keep columns in a logical order
KEEP_COLS = [
    # Identifiers
    "contract_id", "group_id", "country",
    # Dates
    "publication_date", "bid_deadline",
    # Text
    "title", "description",
    # Contract metadata
    "supply_type", "cpv_main", "lot_status",
    # Buyer
    "buyer_id", "buyer_name", "buyer_address", "buyer_email",
    "buyer_contracts_count", "buyer_total_value",
    # OpenTender composite scores
    "score_integrity", "score_transparency", "score_tender",
    # Raw indicator values (kept for reference / feature engineering)
    "ind_single_bid", "ind_advertisement_period",
    "ind_procedure_type", "ind_call_for_tender",
    # Fraud labels
    "single_bid", "short_tender_period", "winner_concentration",
]

df = df[KEEP_COLS].reset_index(drop=True)
print(f"Final shape: {df.shape}")
df.dtypes

## 7. Exploratory Data Analysis

Quick visual checks before saving.

In [ ]:
print("=== Missing value rates ===")
missing = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print(missing[missing > 0].to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Fraud Label Distributions (Ireland)", fontsize=13)

labels = ["single_bid", "short_tender_period", "winner_concentration"]

for ax, col in zip(axes, labels):
    counts = df[col].value_counts(dropna=False).rename({0: "Clean", 1: "Flagged", None: "NaN"})
    counts.plot(kind="bar", ax=ax, color=["steelblue", "tomato", "grey"][:len(counts)])
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Contracts per year
df["year"] = df["publication_date"].dt.year
yearly = df.groupby("year").size()

fig, ax = plt.subplots(figsize=(10, 4))
yearly.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Contracts per Publication Year (Ireland)")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Contracts")
plt.tight_layout()
plt.show()

df.drop(columns="year", inplace=True)

In [ ]:
# Description length distribution
df["desc_len"] = df["description"].str.split().str.len()

fig, ax = plt.subplots(figsize=(10, 4))
df["desc_len"].clip(upper=1000).plot(kind="hist", bins=50, ax=ax, color="steelblue")
ax.set_title("Description Length Distribution (word count, clipped at 1000)")
ax.set_xlabel("Words")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

print(df["desc_len"].describe())
df.drop(columns="desc_len", inplace=True)

## 8. Save

In [ ]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print(f"Saved {len(df):,} rows to {OUTPUT_FILE}")